In [ ]:
# 01. Preparando as bibliotecas e carregando os dados
# Pandas organiza os dados em tabelas; NumPy ajuda nos cálculos.
# Seaborn e Matplotlib são usados para montar os gráficos.
# O estilo ggplot define a aparência e rcParams define o tamanho dos gráficos.
# %matplotlib inline mostra os gráficos dentro do notebook Jupyter.
# chained_assignment = None desativa avisos de atribuição encadeada;
# isso não corrige possíveis problemas de atribuição.
# O read_csv carrega o arquivo movies.csv em uma tabela chamada df.
# O CSV deve ficar em data/movies.csv dentro do projeto.

import pandas as pd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib
plt.style.use('ggplot')
from matplotlib.pyplot import figure

%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (12,8)

pd.options.mode.chained_assignment = None



from pathlib import Path

# Localizando data/movies.csv a partir da raiz do projeto ou de notebooks/.
project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
csv_path = project_dir / 'data' / 'movies.csv'
df = pd.read_csv(csv_path)



In [ ]:
# 02. Conhecendo a base
# Exibindo df para observar as colunas e os dados dos filmes.


df

In [ ]:
# 03. Verificando dados ausentes
# Percorro cada coluna e uso isnull() para identificar valores nulos.
# A média dos valores booleanos dá a proporção de dados ausentes.
# Multiplico por 100 e arredondo para mostrar a porcentagem.
# Uma porcentagem exibida como 0% pode esconder valores pequenos pelo arredondamento.


for col in df.columns:
    pct_missing = np.mean(df[col].isnull())
    print('{} - {}%'.format(col, round(pct_missing*100)))

In [ ]:
# 04. Conferindo os tipos das colunas
# Os tipos ajudam a entender quais campos são números e quais são textos.
# Essa conferência também ajuda a perceber se alguma conversão será necessária.


print(df.dtypes)

In [ ]:
# 05. Observando valores fora do padrão na bilheteria
# O boxplot resume a distribuição de gross e destaca possíveis outliers.
# Um valor destacado não é necessariamente um erro: pode ser um filme de grande bilheteria.


df.boxplot(column=['gross'])

In [ ]:
# 06. Conferindo registros duplicados
# O drop_duplicates retorna a tabela sem linhas totalmente repetidas.
# Como não há atribuição nem inplace=True, o df original não é alterado.

df.drop_duplicates()

In [ ]:
# 07. Ordenando os filmes pela bilheteria
# ascending=False coloca as maiores bilheterias primeiro.
# inplace=False retorna uma tabela ordenada sem modificar df.


df.sort_values(by=['gross'], inplace=False, ascending=False)


In [ ]:
# 08. Comparando bilheteria e orçamento
# O regplot mostra os pontos e uma linha de regressão para observar a tendência.
# Aqui, gross fica no eixo x e budget no eixo y.

sns.regplot(x="gross", y="budget", data=df)

In [ ]:
# 09. Comparando nota e bilheteria
# Quero observar se filmes com notas maiores também tendem a arrecadar mais.
# score representa a nota e gross representa a bilheteria.

sns.regplot(x="score", y="gross", data=df)

In [ ]:
# 10. Calculando a correlação de Pearson
# Pearson mede a associação linear entre variáveis numéricas, de -1 a 1.
# Perto de 1: associação positiva; perto de -1: negativa; perto de 0: pouca associação linear.
# Correlação não prova que uma variável causa a outra.
# numeric_only=True seleciona as colunas numéricas para o cálculo.


df.corr(numeric_only=True, method ='pearson')

In [ ]:
# 11. Comparando com a correlação de Kendall
# Kendall compara a concordância da ordenação dos valores entre as variáveis.
# numeric_only=True mantém apenas as colunas numéricas.

df.corr(numeric_only=True, method ='kendall')

In [ ]:
# 12. Comparando com a correlação de Spearman
# Spearman trabalha com a posição dos valores e ajuda a observar relações monotônicas:
# quando uma variável aumenta, a outra tende a aumentar ou diminuir.
# numeric_only=True mantém apenas as colunas numéricas.

df.corr(numeric_only=True, method ='spearman')

In [ ]:
# 13. Visualizando a matriz de correlação
# corr() usa Pearson por padrão e guarda o resultado em correlation_matrix.
# O heatmap usa cores para facilitar a comparação; annot=True mostra os números.
# Os títulos e nomes dos eixos foram mantidos como estão na aula.
# plt.show() exibe o gráfico.

correlation_matrix = df.corr(numeric_only=True)

sns.heatmap(correlation_matrix, annot = True)

plt.title("Correlation matrix for Numeric Features")

plt.xlabel("Movie features")

plt.ylabel("Movie features")

plt.show()

In [ ]:
# 14. Experimentando a transformação dos valores em códigos
# apply percorre as colunas e factorize()[0] retorna um código para cada valor distinto.
# Os códigos seguem a ordem de aparecimento; não são números aleatórios.
# Esse comando transforma todas as colunas, inclusive as que já são numéricas.
# Por isso, a correlação resultante é entre os códigos, não os valores originais.
# Para categorias sem ordem, esses códigos não representam uma escala de medida.


df.apply(lambda x: x.factorize()[0]).corr(method='pearson')

In [ ]:
# 15. Visualizando a correlação entre os códigos
# Repito a transformação com factorize e monto outro mapa de calor.
# Esse gráfico é uma exploração dos códigos; não devo interpretar uma associação
# entre nomes de empresas ou pessoas como se os códigos fossem medidas reais.

correlation_matrix = df.apply(lambda x: x.factorize()[0]).corr(method='pearson')

sns.heatmap(correlation_matrix, annot = True)

plt.title("Correlation matrix for Movies")

plt.xlabel("Movie features")

plt.ylabel("Movie features")

plt.show()

In [ ]:
# 17. Organizando as correlações em pares
# Calculo novamente a matriz entre os códigos.
# unstack transforma a matriz em uma série com pares de colunas e seus valores.
# Isso facilita ordenar e filtrar as correlações depois.

correlation_mat = df.apply(lambda x: x.factorize()[0]).corr()

corr_pairs = correlation_mat.unstack()

print(corr_pairs)

In [ ]:
# 18. Ordenando os pares de correlação
# sort_values organiza os valores em ordem crescente por padrão.
# kind="quicksort" escolhe o algoritmo de ordenação.

sorted_pairs = corr_pairs.sort_values(kind="quicksort")

print(sorted_pairs)

In [ ]:
# 19. Filtrando associações com magnitude acima de 0,5
# abs considera tanto correlações positivas quanto negativas.
# O filtro também mantém a correlação de cada coluna com ela mesma e pares repetidos.
# Como os valores foram codificados, preciso considerar as limitações dessa transformação.


strong_pairs = sorted_pairs[abs(sorted_pairs) > 0.5]

print(strong_pairs)

In [ ]:
# 20. Somando a bilheteria por empresa
# Agrupo os filmes por company e somo gross dentro de cada grupo.
# Ordeno do maior para o menor e seleciono as primeiras 15 empresas.
# astype('int64') converte os resultados para inteiros, descartando casas decimais.
# Estou somando bilheteria dos filmes da base, não calculando lucro das empresas.


CompanyGrossSum = df.groupby('company')[["gross"]].sum()

CompanyGrossSumSorted = CompanyGrossSum.sort_values('gross', ascending = False)[:15]

CompanyGrossSumSorted = CompanyGrossSumSorted['gross'].astype('int64') 

CompanyGrossSumSorted

In [ ]:
# 21. Extraindo o ano da data de lançamento
# Extraio o ano com quatro dígitos de released, inclusive em datas por extenso.
# A coluna criada é Year, diferente da coluna year que já existe na base.

df['Year'] = df['released'].astype(str).str.extract(r'(\d{4})', expand=False)
df

In [ ]:
# 22. Agrupando a bilheteria por empresa e ano
# Uso company e year para observar a soma da bilheteria de cada empresa em cada ano.
# Este agrupamento usa year, não a coluna Year criada no passo anterior.



df.groupby(['company', 'year'])[["gross"]].sum()

In [ ]:
# 23. Selecionando as 15 maiores somas por empresa e ano
# Cada resultado representa uma combinação de empresa e ano.
# A ordenação considera gross, company e year, todos em ordem decrescente.
# Depois seleciono os 15 primeiros e converto a bilheteria para inteiro.


CompanyGrossSum = df.groupby(['company', 'year'])[["gross"]].sum()

CompanyGrossSumSorted = CompanyGrossSum.sort_values(['gross','company','year'], ascending = False)[:15]

CompanyGrossSumSorted = CompanyGrossSumSorted['gross'].astype('int64') 

CompanyGrossSumSorted

In [ ]:
# 24. Retomando o ranking geral das empresas
# Agrupo apenas por empresa para somar a bilheteria de todos os anos da base.
# Ordeno por gross e company e seleciono as 15 primeiras.
# Essa atribuição substitui os resultados anteriores dessas mesmas variáveis.


CompanyGrossSum = df.groupby(['company'])[["gross"]].sum()

CompanyGrossSumSorted = CompanyGrossSum.sort_values(['gross','company'], ascending = False)[:15]

CompanyGrossSumSorted = CompanyGrossSumSorted['gross'].astype('int64') 

CompanyGrossSumSorted

In [ ]:
# 25. Criando um gráfico de dispersão de orçamento e bilheteria
# Cada ponto representa um filme; alpha=0.5 deixa os pontos semitransparentes.
# Os dados colocam budget no eixo x e gross no eixo y.
# Os rótulos acompanham os dados: orçamento em x e bilheteria em y.

plt.scatter(x=df['budget'], y=df['gross'], alpha=0.5)
plt.title('Budget vs Gross Earnings')
plt.xlabel('Budget for Film')
plt.ylabel('Gross Earnings')
plt.show()

In [ ]:
# 26. Conferindo a tabela antes da conversão das categorias
# Exibindo df novamente para acompanhar o estado dos dados.

df


In [ ]:
# 27. Convertendo colunas de texto em códigos de categorias
# Percorro as colunas e converto as do tipo object para category.
# cat.codes substitui cada categoria por um número; valores ausentes recebem -1.
# Uso copy() para manter o df original separado da tabela com códigos.
# Os códigos identificam categorias, mas não medem distância ou importância entre elas.

df_numerized = df.copy()


for col_name in df_numerized.columns:
    if(df_numerized[col_name].dtype == 'object'):
        df_numerized[col_name]= df_numerized[col_name].astype('category')
        df_numerized[col_name] = df_numerized[col_name].cat.codes
        
df_numerized

In [ ]:
# 28. Calculando Pearson depois da conversão
# Agora calculo a correlação incluindo as colunas convertidas para números.
# As correlações envolvendo códigos de categorias sem ordem exigem cuidado na interpretação.

df_numerized.corr(method='pearson')

In [ ]:
# 29. Visualizando a matriz da tabela convertida
# O mapa de calor reúne as correlações de Pearson de df_numerized.
# annot=True mostra os coeficientes dentro de cada quadrado.

correlation_matrix = df_numerized.corr(method='pearson')

sns.heatmap(correlation_matrix, annot = True)

plt.title("Correlation matrix for Movies")

plt.xlabel("Movie features")

plt.ylabel("Movie features")

plt.show()

In [ ]:
# 30. Repetindo a conversão de colunas object no df
# Este trecho também está no notebook original e foi mantido.
# Se todas as colunas object já tiverem sido convertidas, ele não fará novas conversões.

for col_name in df.columns:
    if(df[col_name].dtype == 'object'):
        df[col_name]= df[col_name].astype('category')
        df[col_name] = df[col_name].cat.codes

In [ ]:
# Convertendo apenas as colunas que ainda são do tipo category.
# A seleção evita usar uma variável indefinida ou acessar .cat em inteiros.
cat_columns = df.select_dtypes(include=['category']).columns
if len(cat_columns) > 0:
    df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

df


In [ ]:
# 32. Comparando bilheteria por classificação com swarmplot
# O swarmplot distribui os pontos de cada grupo tentando evitar sobreposição.
# Com muitos filmes, essa visualização pode ficar lenta ou muito cheia.
# Se os passos anteriores foram executados, rating já pode estar em códigos numéricos.

# Usando uma amostra reprodutível para evitar um gráfico muito lento.
# O stripplot seguinte usa a base completa.
swarm_sample = df.sample(n=min(500, len(df)), random_state=42)
sns.swarmplot(x="rating", y="gross", data=swarm_sample, size=2)


In [ ]:
# 33. Comparando a mesma relação com stripplot
# O stripplot também mostra pontos por classificação, mas pode apresentar sobreposição.
# Comparo essa visualização com a anterior para estudar formas de mostrar os mesmos dados.

sns.stripplot(x="rating", y="gross", data=df)